# Task N — AUG OOS vs Full-Sample IS Benchmark Comparison

This notebook provides the final comparison and assessment of the AUG / SHFE Gold Futures replication.

It consolidates the three finalized AUG analyses:

- **Task K:** rolling walk-forward out-of-sample evaluation using a 4-year in-sample optimization window and a 3-month OOS horizon;
- **Task L:** full-sample optimization using the entire available AUG history as one in-sample hindsight benchmark;
- **Task M:** walk-forward sensitivity analysis across alternative \(T/\tau\) specifications.

The purpose of Task N is not to perform another parameter optimization. Instead, it evaluates how the finalized rolling OOS results compare with the full-sample hindsight benchmark and interprets the result together with the robustness and diagnostic evidence established in Tasks K–M.

Because Task K and Task L cover different historical periods and use fundamentally different evaluation designs, their raw cumulative profits are descriptive rather than directly comparable. Greater emphasis is therefore placed on normalized performance measures, trade-level characteristics, parameter behavior, and known robustness limitations.

The full-sample Task L result is treated strictly as an **in-sample hindsight benchmark**, not as an OOS competitor.

No new strategy specification is selected in Task N.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option(
    "display.float_format",
    "{:,.6f}".format
)


# ============================================================
# Finalized Task K
# Rolling Walk-Forward OOS
# ============================================================

TASK_K = {
    "Label":
        "Task K — Rolling Walk-Forward OOS",

    "Evaluation_Type":
        "Rolling OOS",

    "Sample_Start":
        pd.Timestamp("2022-07-01 09:05:00"),

    "Sample_End":
        pd.Timestamp("2026-03-31 15:00:00"),

    "Starting_Equity":
        100_000.0,

    "Ending_Equity":
        955_221.26,

    "Net_Profit":
        855_221.26,

    "Total_Return":
        8.552213,

    "CAGR":
        0.825766,

    "Max_Drawdown_CNY":
        -17_014.1775,

    "Max_Drawdown_Pct":
        -0.099059,

    "Daily_Sharpe":
        3.608593,

    "Calmar":
        8.336143,

    "Completed_Trades":
        173,

    "Win_Rate":
        0.583815,

    "Profit_Factor":
        7.186075,

    "Expectancy":
        4_684.539,

    "Payoff_Ratio":
        5.122747,

    "Average_Holding_Bars":
        77.7457,

    "Median_Holding_Bars":
        58.0,

    "Long_Trades":
        123,

    "Short_Trades":
        50,

    "Median_ChnLen":
        1920.0,

    "Unique_ChnLen":
        8,

    "Median_StpPct":
        0.005,

    "StpPct_Lower_Bound_Share":
        1.0,

    "OOS_Windows":
        15,

    "Positive_OOS_Windows":
        15,

    "Negative_OOS_Windows":
        0,

    "Largest_Quarter_Share":
        0.410604,

    "Top3_Quarter_Share":
        0.631136,

    "Session_Gap_PnL_Share":
        0.766696,

    "NonGap_Diagnostic_CAGR":
        0.339955,

    "NonGap_Diagnostic_MDD_Pct":
        -0.125938,

    "NonGap_Diagnostic_Calmar":
        2.699387,
}


# ============================================================
# Finalized Task L
# Full-Sample Hindsight IS Benchmark
# ============================================================

TASK_L = {
    "Label":
        "Task L — Full-Sample IS Benchmark",

    "Evaluation_Type":
        "Full-Sample IS / Hindsight",

    "Sample_Start":
        pd.Timestamp("2018-05-03 09:05:00"),

    "Sample_End":
        pd.Timestamp("2026-04-10 15:00:00"),

    "Starting_Equity":
        100_000.0,

    "Ending_Equity":
        1_515_095.395,

    "Net_Profit":
        1_415_095.395,

    "Total_Return":
        14.150954,

    "CAGR":
        0.408358,

    "Max_Drawdown_CNY":
        -16_520.4825,

    "Max_Drawdown_Pct":
        -0.069410,

    "Daily_Sharpe":
        3.687327,

    "Calmar":
        5.883255,

    "Completed_Trades":
        417,

    "Win_Rate":
        0.606715,

    "Profit_Factor":
        7.415238,

    "Expectancy":
        3_393.514137,

    "Payoff_Ratio":
        4.806716,

    "Average_Holding_Bars":
        85.6259,

    "Median_Holding_Bars":
        68.0,

    "Long_Trades":
        253,

    "Short_Trades":
        164,

    "Optimal_ChnLen":
        710,

    "Optimal_StpPct":
        0.005,

    "StpPct_Lower_Bound":
        True,

    "ChnLen_Lower_Bound":
        False,

    "ChnLen_Upper_Bound":
        False,
}


# ============================================================
# Finalized Task M robustness evidence
# ============================================================

TASK_M = {
    "Specifications":
        6,

    "T_Years":
        (4, 5, 6),

    "Tau_Months":
        (3, 6),

    "Common_Start":
        pd.Timestamp("2024-07-01 09:05:00"),

    "Common_End":
        pd.Timestamp("2025-12-31 15:00:00"),

    "Common_Bars":
        26_496,

    "CAGR_Min":
        1.676813,

    "CAGR_Max":
        1.830535,

    "Sharpe_Min":
        3.979571,

    "Sharpe_Max":
        4.301115,

    "MDD_Pct_Min":
        -0.141416,

    "MDD_Pct_Max":
        -0.141416,

    "Calmar_Min":
        11.857294,

    "Calmar_Max":
        12.944313,

    "Median_ChnLen_Min":
        640.0,

    "Median_ChnLen_Max":
        1920.0,

    "StpPct_Lower_Bound_Share_Min":
        1.0,

    "StpPct_Lower_Bound_Share_Max":
        1.0,

    "Minimum_OOS_Windows":
        3,

    "Maximum_OOS_Windows":
        15,
}


# ============================================================
# Configuration audit
# ============================================================

print("=" * 88)
print("TASK N — FINALIZED AUG INPUT AUDIT")
print("=" * 88)

print()
print("TASK K — ROLLING OOS")
print(
    f"Period       : "
    f"{TASK_K['Sample_Start']} "
    f"to {TASK_K['Sample_End']}"
)
print(
    f"Ending equity: "
    f"{TASK_K['Ending_Equity']:,.2f} CNY"
)
print(
    f"CAGR         : "
    f"{100 * TASK_K['CAGR']:.2f}%"
)
print(
    f"Sharpe       : "
    f"{TASK_K['Daily_Sharpe']:.4f}"
)
print(
    f"MDD          : "
    f"{100 * TASK_K['Max_Drawdown_Pct']:.2f}%"
)
print(
    f"Calmar       : "
    f"{TASK_K['Calmar']:.4f}"
)


print()
print("TASK L — FULL-SAMPLE HINDSIGHT IS")
print(
    f"Period       : "
    f"{TASK_L['Sample_Start']} "
    f"to {TASK_L['Sample_End']}"
)
print(
    f"Ending equity: "
    f"{TASK_L['Ending_Equity']:,.2f} CNY"
)
print(
    f"CAGR         : "
    f"{100 * TASK_L['CAGR']:.2f}%"
)
print(
    f"Sharpe       : "
    f"{TASK_L['Daily_Sharpe']:.4f}"
)
print(
    f"MDD          : "
    f"{100 * TASK_L['Max_Drawdown_Pct']:.2f}%"
)
print(
    f"Calmar       : "
    f"{TASK_L['Calmar']:.4f}"
)


print()
print("TASK M — WINDOW SENSITIVITY")
print(
    f"Specifications: "
    f"{TASK_M['Specifications']}"
)
print(
    f"Common period: "
    f"{TASK_M['Common_Start']} "
    f"to {TASK_M['Common_End']}"
)
print(
    f"CAGR range   : "
    f"{100 * TASK_M['CAGR_Min']:.2f}% "
    f"to "
    f"{100 * TASK_M['CAGR_Max']:.2f}%"
)
print(
    f"Sharpe range : "
    f"{TASK_M['Sharpe_Min']:.4f} "
    f"to "
    f"{TASK_M['Sharpe_Max']:.4f}"
)
print(
    f"Calmar range : "
    f"{TASK_M['Calmar_Min']:.4f} "
    f"to "
    f"{TASK_M['Calmar_Max']:.4f}"
)


# ============================================================
# Basic internal consistency checks
# ============================================================

assert np.isclose(
    TASK_K["Ending_Equity"]
    -
    TASK_K["Starting_Equity"],
    TASK_K["Net_Profit"],
    rtol=0.0,
    atol=1e-6
)

assert np.isclose(
    TASK_L["Ending_Equity"]
    -
    TASK_L["Starting_Equity"],
    TASK_L["Net_Profit"],
    rtol=0.0,
    atol=1e-6
)

assert TASK_K["Evaluation_Type"] == "Rolling OOS"

assert (
    TASK_L["Evaluation_Type"]
    ==
    "Full-Sample IS / Hindsight"
)

assert (
    TASK_M["StpPct_Lower_Bound_Share_Min"]
    ==
    1.0
)

assert (
    TASK_M["StpPct_Lower_Bound_Share_Max"]
    ==
    1.0
)


print()
print(
    "ALL FINALIZED TASK K / L / M "
    "INPUT CHECKS PASSED."
)

TASK N — FINALIZED AUG INPUT AUDIT

TASK K — ROLLING OOS
Period       : 2022-07-01 09:05:00 to 2026-03-31 15:00:00
Ending equity: 955,221.26 CNY
CAGR         : 82.58%
Sharpe       : 3.6086
MDD          : -9.91%
Calmar       : 8.3361

TASK L — FULL-SAMPLE HINDSIGHT IS
Period       : 2018-05-03 09:05:00 to 2026-04-10 15:00:00
Ending equity: 1,515,095.40 CNY
CAGR         : 40.84%
Sharpe       : 3.6873
MDD          : -6.94%
Calmar       : 5.8833

TASK M — WINDOW SENSITIVITY
Specifications: 6
Common period: 2024-07-01 09:05:00 to 2025-12-31 15:00:00
CAGR range   : 167.68% to 183.05%
Sharpe range : 3.9796 to 4.3011
Calmar range : 11.8573 to 12.9443

ALL FINALIZED TASK K / L / M INPUT CHECKS PASSED.


## 1. Headline Performance Comparison — Rolling OOS vs Full-Sample Hindsight IS

This section compares the finalized Task K rolling walk-forward OOS results with the finalized Task L full-sample hindsight IS benchmark.

The two evaluations should not be interpreted as directly competing backtests:

- **Task K** evaluates parameters selected using only prior in-sample information and records subsequent rolling OOS performance.
- **Task L** optimizes one parameter pair using the entire AUG history and therefore represents a hindsight in-sample benchmark.

The evaluation periods are also different. Consequently, ending equity, cumulative return, and net profit are reported descriptively but are not used as the primary basis for judging OOS deterioration.

The main comparison instead focuses on normalized risk-adjusted measures and trade-level characteristics, including CAGR, maximum drawdown, Sharpe ratio, Calmar ratio, win rate, profit factor, expectancy, and payoff ratio.

For selected metrics, an **OOS / IS benchmark ratio** is also reported. This ratio is a descriptive retention measure rather than a formal statistical test.

In [2]:
# ------------------------------------------------------------
# 1.1 Build headline comparison table
# ------------------------------------------------------------

headline_comparison = pd.DataFrame(
    {
        "Task K — Rolling OOS": {
            "Evaluation Type":
                TASK_K["Evaluation_Type"],

            "Sample Start":
                TASK_K["Sample_Start"],

            "Sample End":
                TASK_K["Sample_End"],

            "Starting Equity (CNY)":
                TASK_K["Starting_Equity"],

            "Ending Equity (CNY)":
                TASK_K["Ending_Equity"],

            "Net Profit (CNY)":
                TASK_K["Net_Profit"],

            "Total Return":
                TASK_K["Total_Return"],

            "CAGR":
                TASK_K["CAGR"],

            "Maximum Drawdown (CNY)":
                TASK_K["Max_Drawdown_CNY"],

            "Maximum Drawdown (%)":
                TASK_K["Max_Drawdown_Pct"],

            "Daily Sharpe":
                TASK_K["Daily_Sharpe"],

            "Calmar":
                TASK_K["Calmar"],

            "Completed Trades":
                TASK_K["Completed_Trades"],

            "Win Rate":
                TASK_K["Win_Rate"],

            "Profit Factor":
                TASK_K["Profit_Factor"],

            "Expectancy (CNY/trade)":
                TASK_K["Expectancy"],

            "Payoff Ratio":
                TASK_K["Payoff_Ratio"],
        },

        "Task L — Full-Sample IS": {
            "Evaluation Type":
                TASK_L["Evaluation_Type"],

            "Sample Start":
                TASK_L["Sample_Start"],

            "Sample End":
                TASK_L["Sample_End"],

            "Starting Equity (CNY)":
                TASK_L["Starting_Equity"],

            "Ending Equity (CNY)":
                TASK_L["Ending_Equity"],

            "Net Profit (CNY)":
                TASK_L["Net_Profit"],

            "Total Return":
                TASK_L["Total_Return"],

            "CAGR":
                TASK_L["CAGR"],

            "Maximum Drawdown (CNY)":
                TASK_L["Max_Drawdown_CNY"],

            "Maximum Drawdown (%)":
                TASK_L["Max_Drawdown_Pct"],

            "Daily Sharpe":
                TASK_L["Daily_Sharpe"],

            "Calmar":
                TASK_L["Calmar"],

            "Completed Trades":
                TASK_L["Completed_Trades"],

            "Win Rate":
                TASK_L["Win_Rate"],

            "Profit Factor":
                TASK_L["Profit_Factor"],

            "Expectancy (CNY/trade)":
                TASK_L["Expectancy"],

            "Payoff Ratio":
                TASK_L["Payoff_Ratio"],
        },
    }
)


print("=" * 100)
print("TASK N — HEADLINE PERFORMANCE COMPARISON")
print("=" * 100)
display(headline_comparison)


# ------------------------------------------------------------
# 1.2 OOS / IS benchmark retention measures
#
# These are descriptive ratios only.
# They are NOT formal statistical tests because Task K and
# Task L use different evaluation periods and methodologies.
# ------------------------------------------------------------

retention_metrics = pd.DataFrame(
    {
        "Metric": [
            "CAGR",
            "Daily Sharpe",
            "Calmar",
            "Win Rate",
            "Profit Factor",
            "Expectancy",
            "Payoff Ratio",
        ],

        "Task K OOS": [
            TASK_K["CAGR"],
            TASK_K["Daily_Sharpe"],
            TASK_K["Calmar"],
            TASK_K["Win_Rate"],
            TASK_K["Profit_Factor"],
            TASK_K["Expectancy"],
            TASK_K["Payoff_Ratio"],
        ],

        "Task L IS Benchmark": [
            TASK_L["CAGR"],
            TASK_L["Daily_Sharpe"],
            TASK_L["Calmar"],
            TASK_L["Win_Rate"],
            TASK_L["Profit_Factor"],
            TASK_L["Expectancy"],
            TASK_L["Payoff_Ratio"],
        ],
    }
)

retention_metrics["OOS / IS Ratio"] = (
    retention_metrics["Task K OOS"]
    /
    retention_metrics["Task L IS Benchmark"]
)

retention_metrics["Relative Difference"] = (
    retention_metrics["Task K OOS"]
    /
    retention_metrics["Task L IS Benchmark"]
    - 1.0
)


print()
print("=" * 100)
print("DESCRIPTIVE OOS / IS BENCHMARK RETENTION")
print("=" * 100)
display(retention_metrics)


# ------------------------------------------------------------
# 1.3 Drawdown comparison
#
# Use absolute percentage drawdown so that a larger ratio means
# Task K experienced a deeper drawdown than Task L.
# ------------------------------------------------------------

k_abs_mdd = abs(
    TASK_K["Max_Drawdown_Pct"]
)

l_abs_mdd = abs(
    TASK_L["Max_Drawdown_Pct"]
)

mdd_ratio = (
    k_abs_mdd
    /
    l_abs_mdd
)

mdd_difference = (
    k_abs_mdd
    -
    l_abs_mdd
)


print()
print("=" * 100)
print("DRAWDOWN COMPARISON")
print("=" * 100)

print(
    f"Task K OOS MDD              : "
    f"{100 * TASK_K['Max_Drawdown_Pct']:.2f}%"
)

print(
    f"Task L hindsight IS MDD     : "
    f"{100 * TASK_L['Max_Drawdown_Pct']:.2f}%"
)

print(
    f"OOS / IS absolute MDD ratio : "
    f"{mdd_ratio:.4f}"
)

print(
    f"Absolute MDD difference     : "
    f"{100 * mdd_difference:.2f} percentage points"
)


# ------------------------------------------------------------
# 1.4 Key descriptive comparisons
# ------------------------------------------------------------

sharpe_retention = (
    TASK_K["Daily_Sharpe"]
    /
    TASK_L["Daily_Sharpe"]
)

pf_retention = (
    TASK_K["Profit_Factor"]
    /
    TASK_L["Profit_Factor"]
)

win_rate_retention = (
    TASK_K["Win_Rate"]
    /
    TASK_L["Win_Rate"]
)

expectancy_ratio = (
    TASK_K["Expectancy"]
    /
    TASK_L["Expectancy"]
)

payoff_ratio_comparison = (
    TASK_K["Payoff_Ratio"]
    /
    TASK_L["Payoff_Ratio"]
)


print()
print("=" * 100)
print("KEY OOS / IS DESCRIPTIVE RATIOS")
print("=" * 100)

print(
    f"Sharpe retention            : "
    f"{sharpe_retention:.4f}"
)

print(
    f"Profit-factor retention      : "
    f"{pf_retention:.4f}"
)

print(
    f"Win-rate retention           : "
    f"{win_rate_retention:.4f}"
)

print(
    f"Expectancy ratio             : "
    f"{expectancy_ratio:.4f}"
)

print(
    f"Payoff-ratio comparison      : "
    f"{payoff_ratio_comparison:.4f}"
)


# ------------------------------------------------------------
# 1.5 Internal validation
# ------------------------------------------------------------

comparison_checks = pd.DataFrame(
    {
        "Validation Check": [
            "Task K identified as rolling OOS",
            "Task L identified as hindsight IS",
            "Both strategies start from 100,000 CNY",
            "Task K Sharpe remains positive",
            "Task L Sharpe remains positive",
            "Task K profit factor remains above 1",
            "Task L profit factor remains above 1",
            "Task K expectancy remains positive",
            "Task L expectancy remains positive",
        ],

        "Passed": [
            TASK_K["Evaluation_Type"] == "Rolling OOS",

            (
                TASK_L["Evaluation_Type"]
                ==
                "Full-Sample IS / Hindsight"
            ),

            np.isclose(
                TASK_K["Starting_Equity"],
                TASK_L["Starting_Equity"],
                rtol=0.0,
                atol=1e-12
            ),

            TASK_K["Daily_Sharpe"] > 0.0,

            TASK_L["Daily_Sharpe"] > 0.0,

            TASK_K["Profit_Factor"] > 1.0,

            TASK_L["Profit_Factor"] > 1.0,

            TASK_K["Expectancy"] > 0.0,

            TASK_L["Expectancy"] > 0.0,
        ],
    }
)


print()
print("=" * 100)
print("SECTION 1 VALIDATION")
print("=" * 100)
display(comparison_checks)

assert comparison_checks["Passed"].all()


# ------------------------------------------------------------
# 1.6 Interpretation
# ------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 1 INTERPRETATION")
print("=" * 100)

print(
    "1. Task K and Task L are not directly competing backtests: "
    "Task K is rolling OOS, whereas Task L is a full-sample "
    "hindsight IS benchmark."
)

print(
    "2. Raw cumulative profit and ending equity are therefore "
    "descriptive only because the two evaluations cover "
    "different historical periods."
)

print(
    "3. The Task K OOS Sharpe ratio and profit factor remain "
    "broadly comparable to their Task L hindsight-IS levels, "
    "so the historical OOS result does not exhibit a collapse "
    "in these risk-adjusted and trade-level measures."
)

print(
    "4. Task K experiences a deeper percentage drawdown than "
    "Task L, showing that strong OOS profitability is accompanied "
    "by greater realized downside risk."
)

print(
    "5. These comparisons are descriptive evidence only and do "
    "not establish that OOS performance is superior to the "
    "hindsight benchmark or prove the absence of overfitting."
)

print()
print("SECTION 1 VALIDATION PASSED.")

TASK N — HEADLINE PERFORMANCE COMPARISON


,Task K — Rolling OOS,Task L — Full-Sample IS
Evaluation Type,Rolling OOS,Full-Sample IS / Hindsight
Sample Start,2022-07-01 09:05:00,2018-05-03 09:05:00
Sample End,2026-03-31 15:00:00,2026-04-10 15:00:00
Starting Equity (CNY),"100,000.000000","100,000.000000"
Ending Equity (CNY),"955,221.260000","1,515,095.395000"
Net Profit (CNY),"855,221.260000","1,415,095.395000"
Total Return,8.552213,14.150954
CAGR,0.825766,0.408358
Maximum Drawdown (CNY),"-17,014.177500","-16,520.482500"
Maximum Drawdown (%),-0.099059,-0.069410



DESCRIPTIVE OOS / IS BENCHMARK RETENTION


,Metric,Task K OOS,Task L IS Benchmark,OOS / IS Ratio,Relative Difference
0,CAGR,0.825766,0.408358,2.022162,1.022162
1,Daily Sharpe,3.608593,3.687327,0.978647,-0.021353
2,Calmar,8.336143,5.883255,1.416927,0.416927
3,Win Rate,0.583815,0.606715,0.962256,-0.037744
4,Profit Factor,7.186075,7.415238,0.969096,-0.030904
5,Expectancy,"4,684.539000","3,393.514137",1.380439,0.380439
6,Payoff Ratio,5.122747,4.806716,1.065748,0.065748



DRAWDOWN COMPARISON
Task K OOS MDD              : -9.91%
Task L hindsight IS MDD     : -6.94%
OOS / IS absolute MDD ratio : 1.4272
Absolute MDD difference     : 2.96 percentage points

KEY OOS / IS DESCRIPTIVE RATIOS
Sharpe retention            : 0.9786
Profit-factor retention      : 0.9691
Win-rate retention           : 0.9623
Expectancy ratio             : 1.3804
Payoff-ratio comparison      : 1.0657

SECTION 1 VALIDATION


,Validation Check,Passed
0,Task K identified as rolling OOS,True
1,Task L identified as hindsight IS,True
2,"Both strategies start from 100,000 CNY",True
3,Task K Sharpe remains positive,True
4,Task L Sharpe remains positive,True
5,Task K profit factor remains above 1,True
6,Task L profit factor remains above 1,True
7,Task K expectancy remains positive,True
8,Task L expectancy remains positive,True



SECTION 1 INTERPRETATION
1. Task K and Task L are not directly competing backtests: Task K is rolling OOS, whereas Task L is a full-sample hindsight IS benchmark.
2. Raw cumulative profit and ending equity are therefore descriptive only because the two evaluations cover different historical periods.
3. The Task K OOS Sharpe ratio and profit factor remain broadly comparable to their Task L hindsight-IS levels, so the historical OOS result does not exhibit a collapse in these risk-adjusted and trade-level measures.
4. Task K experiences a deeper percentage drawdown than Task L, showing that strong OOS profitability is accompanied by greater realized downside risk.
5. These comparisons are descriptive evidence only and do not establish that OOS performance is superior to the hindsight benchmark or prove the absence of overfitting.

SECTION 1 VALIDATION PASSED.


## 2. Trade-Level Quality Comparison

Headline portfolio statistics alone do not reveal whether the underlying trade-level edge deteriorates materially out of sample.

This section therefore compares the finalized Task K OOS trade characteristics with the Task L full-sample hindsight IS benchmark.

The comparison focuses on:

- win rate;
- profit factor;
- average expectancy per completed trade;
- payoff ratio;
- holding-period characteristics;
- long / short participation.

Because Task K and Task L cover different historical periods and contain different numbers of trades, trade counts are interpreted descriptively rather than as direct performance rankings.

The purpose is to determine whether the historical OOS result retains a broadly similar trade-quality structure to the hindsight benchmark, rather than whether one sample produces more total trades or total profit.

In [3]:
# ------------------------------------------------------------
# 2.1 Build trade-quality comparison table
# ------------------------------------------------------------

trade_comparison = pd.DataFrame(
    {
        "Metric": [
            "Completed Trades",
            "Win Rate",
            "Profit Factor",
            "Expectancy (CNY/trade)",
            "Payoff Ratio",
            "Average Holding Bars",
            "Median Holding Bars",
            "Long Trades",
            "Short Trades",
        ],

        "Task K — Rolling OOS": [
            TASK_K["Completed_Trades"],
            TASK_K["Win_Rate"],
            TASK_K["Profit_Factor"],
            TASK_K["Expectancy"],
            TASK_K["Payoff_Ratio"],
            TASK_K["Average_Holding_Bars"],
            TASK_K["Median_Holding_Bars"],
            TASK_K["Long_Trades"],
            TASK_K["Short_Trades"],
        ],

        "Task L — Full-Sample IS": [
            TASK_L["Completed_Trades"],
            TASK_L["Win_Rate"],
            TASK_L["Profit_Factor"],
            TASK_L["Expectancy"],
            TASK_L["Payoff_Ratio"],
            TASK_L["Average_Holding_Bars"],
            TASK_L["Median_Holding_Bars"],
            TASK_L["Long_Trades"],
            TASK_L["Short_Trades"],
        ],
    }
)


# ------------------------------------------------------------
# 2.2 Compute relative OOS / IS trade-quality measures
# ------------------------------------------------------------

ratio_metrics = {
    "Win Rate Retention":
        TASK_K["Win_Rate"]
        /
        TASK_L["Win_Rate"],

    "Profit Factor Retention":
        TASK_K["Profit_Factor"]
        /
        TASK_L["Profit_Factor"],

    "Expectancy Ratio":
        TASK_K["Expectancy"]
        /
        TASK_L["Expectancy"],

    "Payoff Ratio Comparison":
        TASK_K["Payoff_Ratio"]
        /
        TASK_L["Payoff_Ratio"],

    "Average Holding Ratio":
        TASK_K["Average_Holding_Bars"]
        /
        TASK_L["Average_Holding_Bars"],

    "Median Holding Ratio":
        TASK_K["Median_Holding_Bars"]
        /
        TASK_L["Median_Holding_Bars"],
}


ratio_table = pd.DataFrame(
    {
        "Metric": list(ratio_metrics.keys()),
        "OOS / IS Ratio": list(ratio_metrics.values()),
    }
)


# ------------------------------------------------------------
# 2.3 Directional participation
# ------------------------------------------------------------

task_k_long_share = (
    TASK_K["Long_Trades"]
    /
    TASK_K["Completed_Trades"]
)

task_k_short_share = (
    TASK_K["Short_Trades"]
    /
    TASK_K["Completed_Trades"]
)

task_l_long_share = (
    TASK_L["Long_Trades"]
    /
    TASK_L["Completed_Trades"]
)

task_l_short_share = (
    TASK_L["Short_Trades"]
    /
    TASK_L["Completed_Trades"]
)


direction_table = pd.DataFrame(
    {
        "Sample": [
            "Task K — Rolling OOS",
            "Task L — Full-Sample IS",
        ],

        "Long Trades": [
            TASK_K["Long_Trades"],
            TASK_L["Long_Trades"],
        ],

        "Short Trades": [
            TASK_K["Short_Trades"],
            TASK_L["Short_Trades"],
        ],

        "Long Share": [
            task_k_long_share,
            task_l_long_share,
        ],

        "Short Share": [
            task_k_short_share,
            task_l_short_share,
        ],
    }
)


# ------------------------------------------------------------
# 2.4 Display
# ------------------------------------------------------------

print("=" * 100)
print("TASK N — TRADE-LEVEL QUALITY COMPARISON")
print("=" * 100)
display(trade_comparison)


print()
print("=" * 100)
print("TRADE-QUALITY OOS / IS RATIOS")
print("=" * 100)
display(ratio_table)


print()
print("=" * 100)
print("DIRECTIONAL PARTICIPATION")
print("=" * 100)
display(direction_table)


# ------------------------------------------------------------
# 2.5 Additional descriptive differences
# ------------------------------------------------------------

win_rate_difference = (
    TASK_K["Win_Rate"]
    -
    TASK_L["Win_Rate"]
)

pf_difference = (
    TASK_K["Profit_Factor"]
    -
    TASK_L["Profit_Factor"]
)

expectancy_difference = (
    TASK_K["Expectancy"]
    -
    TASK_L["Expectancy"]
)

payoff_difference = (
    TASK_K["Payoff_Ratio"]
    -
    TASK_L["Payoff_Ratio"]
)

avg_holding_difference = (
    TASK_K["Average_Holding_Bars"]
    -
    TASK_L["Average_Holding_Bars"]
)


print()
print("=" * 100)
print("ABSOLUTE TRADE-LEVEL DIFFERENCES")
print("=" * 100)

print(
    f"Win-rate difference        : "
    f"{100 * win_rate_difference:.2f} percentage points"
)

print(
    f"Profit-factor difference   : "
    f"{pf_difference:.4f}"
)

print(
    f"Expectancy difference      : "
    f"{expectancy_difference:,.2f} CNY/trade"
)

print(
    f"Payoff-ratio difference    : "
    f"{payoff_difference:.4f}"
)

print(
    f"Average holding difference : "
    f"{avg_holding_difference:.2f} bars"
)


# ------------------------------------------------------------
# 2.6 Validation
# ------------------------------------------------------------

trade_checks = pd.DataFrame(
    {
        "Validation Check": [
            "Task K completed-trade count positive",
            "Task L completed-trade count positive",
            "Task K long + short trades reconcile",
            "Task L long + short trades reconcile",
            "Task K win rate between 0 and 1",
            "Task L win rate between 0 and 1",
            "Task K profit factor above 1",
            "Task L profit factor above 1",
            "Task K expectancy positive",
            "Task L expectancy positive",
            "Task K payoff ratio above 1",
            "Task L payoff ratio above 1",
        ],

        "Passed": [
            TASK_K["Completed_Trades"] > 0,

            TASK_L["Completed_Trades"] > 0,

            (
                TASK_K["Long_Trades"]
                +
                TASK_K["Short_Trades"]
                ==
                TASK_K["Completed_Trades"]
            ),

            (
                TASK_L["Long_Trades"]
                +
                TASK_L["Short_Trades"]
                ==
                TASK_L["Completed_Trades"]
            ),

            0.0 <= TASK_K["Win_Rate"] <= 1.0,

            0.0 <= TASK_L["Win_Rate"] <= 1.0,

            TASK_K["Profit_Factor"] > 1.0,

            TASK_L["Profit_Factor"] > 1.0,

            TASK_K["Expectancy"] > 0.0,

            TASK_L["Expectancy"] > 0.0,

            TASK_K["Payoff_Ratio"] > 1.0,

            TASK_L["Payoff_Ratio"] > 1.0,
        ],
    }
)


print()
print("=" * 100)
print("SECTION 2 VALIDATION")
print("=" * 100)
display(trade_checks)

assert trade_checks["Passed"].all()


# ------------------------------------------------------------
# 2.7 Interpretation
# ------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 2 INTERPRETATION")
print("=" * 100)

print(
    "1. The rolling OOS sample retains a broadly similar "
    "trade-quality profile to the full-sample hindsight benchmark."
)

print(
    "2. OOS win rate and profit factor are modestly lower than "
    "their hindsight-IS values, but both remain economically strong."
)

print(
    "3. OOS expectancy per completed trade and payoff ratio are "
    "higher than in the full-sample benchmark, indicating that "
    "the OOS result is not driven solely by a higher frequency "
    "of winning trades."
)

print(
    "4. Average and median holding periods remain of a similar "
    "order of magnitude across the two evaluations, suggesting "
    "that the broad trade-duration structure does not change "
    "dramatically out of sample."
)

print(
    "5. Both samples contain substantially more long than short "
    "completed trades, so directional participation should be "
    "recognized when interpreting AUG performance."
)

print(
    "6. These comparisons remain descriptive because Task K "
    "and Task L cover different historical periods."
)

print()
print("SECTION 2 VALIDATION PASSED.")

TASK N — TRADE-LEVEL QUALITY COMPARISON


,Metric,Task K — Rolling OOS,Task L — Full-Sample IS
0,Completed Trades,173.000000,417.000000
1,Win Rate,0.583815,0.606715
2,Profit Factor,7.186075,7.415238
3,Expectancy (CNY/trade),"4,684.539000","3,393.514137"
4,Payoff Ratio,5.122747,4.806716
5,Average Holding Bars,77.745700,85.625900
6,Median Holding Bars,58.000000,68.000000
7,Long Trades,123.000000,253.000000
8,Short Trades,50.000000,164.000000



TRADE-QUALITY OOS / IS RATIOS


,Metric,OOS / IS Ratio
0,Win Rate Retention,0.962256
1,Profit Factor Retention,0.969096
2,Expectancy Ratio,1.380439
3,Payoff Ratio Comparison,1.065748
4,Average Holding Ratio,0.907969
5,Median Holding Ratio,0.852941



DIRECTIONAL PARTICIPATION


,Sample,Long Trades,Short Trades,Long Share,Short Share
0,Task K — Rolling OOS,123,50,0.710983,0.289017
1,Task L — Full-Sample IS,253,164,0.606715,0.393285



ABSOLUTE TRADE-LEVEL DIFFERENCES
Win-rate difference        : -2.29 percentage points
Profit-factor difference   : -0.2292
Expectancy difference      : 1,291.02 CNY/trade
Payoff-ratio difference    : 0.3160
Average holding difference : -7.88 bars

SECTION 2 VALIDATION


,Validation Check,Passed
0,Task K completed-trade count positive,True
1,Task L completed-trade count positive,True
2,Task K long + short trades reconcile,True
3,Task L long + short trades reconcile,True
4,Task K win rate between 0 and 1,True
5,Task L win rate between 0 and 1,True
6,Task K profit factor above 1,True
7,Task L profit factor above 1,True
8,Task K expectancy positive,True
9,Task L expectancy positive,True



SECTION 2 INTERPRETATION
1. The rolling OOS sample retains a broadly similar trade-quality profile to the full-sample hindsight benchmark.
2. OOS win rate and profit factor are modestly lower than their hindsight-IS values, but both remain economically strong.
3. OOS expectancy per completed trade and payoff ratio are higher than in the full-sample benchmark, indicating that the OOS result is not driven solely by a higher frequency of winning trades.
4. Average and median holding periods remain of a similar order of magnitude across the two evaluations, suggesting that the broad trade-duration structure does not change dramatically out of sample.
5. Both samples contain substantially more long than short completed trades, so directional participation should be recognized when interpreting AUG performance.
6. These comparisons remain descriptive because Task K and Task L cover different historical periods.

SECTION 2 VALIDATION PASSED.


## 3. Parameter Behavior and Boundary Risk

The finalized AUG analyses provide consistent evidence about parameter behavior across rolling OOS optimization, full-sample hindsight optimization, and walk-forward sensitivity testing.

For the channel-length parameter, Task K selects multiple values across rolling optimization windows, while Task L identifies a single full-sample optimum. Task M further shows that the median selected channel length changes across alternative walk-forward specifications.

The stop-loss parameter exhibits a different pattern. The professor-specified optimization grid restricts:

\[
\text{StpPct} \in [0.005, 0.100].
\]

Across the finalized AUG analyses:

- every Task K rolling optimization selects `StpPct = 0.005`;
- the Task L full-sample optimum also selects `StpPct = 0.005`;
- every Task M sensitivity specification reports a 100% lower-bound selection frequency.

This should not be interpreted simply as evidence that `0.005` is a precisely identified stable optimum. Because `0.005` is the minimum admissible value in the prescribed grid, the optimization results indicate a persistent **boundary solution**.

The experiment therefore does not identify whether the objective would improve further for stop percentages below 0.005. The prescribed grid is retained unchanged to preserve consistency with the project specification, and no ex-post grid expansion is used to redefine the strategy.

This boundary behavior is treated as a material model and parameter-identification limitation in the final AUG assessment.

In [4]:
# ============================================================
# 3. Parameter Behavior and Boundary Risk
# ============================================================


# ------------------------------------------------------------
# 3.1 Finalized parameter evidence
# ------------------------------------------------------------

parameter_comparison = pd.DataFrame(
    {
        "Evidence": [
            "Task K — Rolling OOS",
            "Task L — Full-Sample IS",
            "Task M — Sensitivity",
        ],

        "ChnLen Result": [
            (
                f"Median = {TASK_K['Median_ChnLen']:.0f}; "
                f"{TASK_K['Unique_ChnLen']} unique values"
            ),

            (
                f"Global optimum = "
                f"{TASK_L['Optimal_ChnLen']}"
            ),

            (
                f"Median range = "
                f"{TASK_M['Median_ChnLen_Min']:.0f} "
                f"to "
                f"{TASK_M['Median_ChnLen_Max']:.0f}"
            ),
        ],

        "StpPct Result": [
            (
                f"Median = "
                f"{TASK_K['Median_StpPct']:.3f}"
            ),

            (
                f"Global optimum = "
                f"{TASK_L['Optimal_StpPct']:.3f}"
            ),

            (
                f"Lower-bound share = "
                f"{100 * TASK_M['StpPct_Lower_Bound_Share_Min']:.0f}% "
                f"to "
                f"{100 * TASK_M['StpPct_Lower_Bound_Share_Max']:.0f}%"
            ),
        ],

        "Lower-Bound Evidence": [
            (
                TASK_K["StpPct_Lower_Bound_Share"]
                == 1.0
            ),

            TASK_L["StpPct_Lower_Bound"],

            (
                TASK_M["StpPct_Lower_Bound_Share_Min"]
                == 1.0
                and
                TASK_M["StpPct_Lower_Bound_Share_Max"]
                == 1.0
            ),
        ],
    }
)


print("=" * 100)
print("TASK N — PARAMETER BEHAVIOR ACROSS TASKS K / L / M")
print("=" * 100)

display(parameter_comparison)


# ------------------------------------------------------------
# 3.2 Explicit stop-grid boundary audit
# ------------------------------------------------------------

PRESCRIBED_STOP_MIN = 0.005
PRESCRIBED_STOP_MAX = 0.100

task_k_stop_at_lower_bound = np.isclose(
    TASK_K["Median_StpPct"],
    PRESCRIBED_STOP_MIN,
    rtol=0.0,
    atol=1e-12
)

task_l_stop_at_lower_bound = np.isclose(
    TASK_L["Optimal_StpPct"],
    PRESCRIBED_STOP_MIN,
    rtol=0.0,
    atol=1e-12
)

task_m_all_stop_at_lower_bound = (
    np.isclose(
        TASK_M["StpPct_Lower_Bound_Share_Min"],
        1.0,
        rtol=0.0,
        atol=1e-12
    )
    and
    np.isclose(
        TASK_M["StpPct_Lower_Bound_Share_Max"],
        1.0,
        rtol=0.0,
        atol=1e-12
    )
)


boundary_summary = pd.DataFrame(
    {
        "Analysis": [
            "Task K — Rolling OOS",
            "Task L — Full-Sample IS",
            "Task M — Sensitivity",
        ],

        "Boundary Evidence": [
            (
                f"{100 * TASK_K['StpPct_Lower_Bound_Share']:.0f}% "
                f"of rolling windows"
            ),

            (
                f"Global optimum "
                f"StpPct = {TASK_L['Optimal_StpPct']:.3f}"
            ),

            (
                f"{100 * TASK_M['StpPct_Lower_Bound_Share_Min']:.0f}% "
                f"to "
                f"{100 * TASK_M['StpPct_Lower_Bound_Share_Max']:.0f}%"
            ),
        ],

        "At Prescribed Lower Bound": [
            task_k_stop_at_lower_bound,
            task_l_stop_at_lower_bound,
            task_m_all_stop_at_lower_bound,
        ],
    }
)


print()
print("=" * 100)
print("STOP-PARAMETER BOUNDARY AUDIT")
print("=" * 100)

print(
    f"Prescribed StpPct grid : "
    f"{PRESCRIBED_STOP_MIN:.3f} "
    f"to "
    f"{PRESCRIBED_STOP_MAX:.3f}"
)

print()

display(boundary_summary)


# ------------------------------------------------------------
# 3.3 Channel-length behavior
# ------------------------------------------------------------

chnlen_summary = pd.DataFrame(
    {
        "Analysis": [
            "Task K — Rolling OOS",
            "Task L — Full-Sample IS",
            "Task M — Sensitivity",
        ],

        "ChnLen Evidence": [
            (
                f"Median {TASK_K['Median_ChnLen']:.0f}; "
                f"{TASK_K['Unique_ChnLen']} unique values"
            ),

            (
                f"Full-sample optimum "
                f"{TASK_L['Optimal_ChnLen']}"
            ),

            (
                f"Median across specifications "
                f"{TASK_M['Median_ChnLen_Min']:.0f}"
                f"–"
                f"{TASK_M['Median_ChnLen_Max']:.0f}"
            ),
        ],

        "Interpretation": [
            "Rolling optimum varies across windows",
            "Single hindsight full-sample optimum",
            "Moderate sensitivity to walk-forward design",
        ],
    }
)


print()
print("=" * 100)
print("CHANNEL-LENGTH BEHAVIOR")
print("=" * 100)

display(chnlen_summary)


# ------------------------------------------------------------
# 3.4 Parameter-identification flags
# ------------------------------------------------------------

parameter_flags = pd.DataFrame(
    {
        "Diagnostic": [
            "Task K stop lower-bound frequency",
            "Task L stop optimum at lower bound",
            "Task M stop lower-bound persistence",
            "Task L ChnLen at lower bound",
            "Task L ChnLen at upper bound",
            "Task M tested alternative T/tau specifications",
        ],

        "Result": [
            TASK_K["StpPct_Lower_Bound_Share"],
            TASK_L["StpPct_Lower_Bound"],
            task_m_all_stop_at_lower_bound,
            TASK_L["ChnLen_Lower_Bound"],
            TASK_L["ChnLen_Upper_Bound"],
            TASK_M["Specifications"],
        ],
    }
)


print()
print("=" * 100)
print("PARAMETER-IDENTIFICATION FLAGS")
print("=" * 100)

display(parameter_flags)


# ------------------------------------------------------------
# 3.5 Validation
# ------------------------------------------------------------

parameter_checks = pd.DataFrame(
    {
        "Validation Check": [
            "Task K StpPct median equals prescribed lower bound",
            "Task K lower-bound frequency equals 100%",
            "Task L optimum StpPct equals prescribed lower bound",
            "Task L lower-bound flag is True",
            "Task M lower-bound share minimum equals 100%",
            "Task M lower-bound share maximum equals 100%",
            "Task L ChnLen is not at lower boundary",
            "Task L ChnLen is not at upper boundary",
            "Task M contains six sensitivity specifications",
        ],

        "Passed": [
            task_k_stop_at_lower_bound,

            np.isclose(
                TASK_K["StpPct_Lower_Bound_Share"],
                1.0,
                rtol=0.0,
                atol=1e-12
            ),

            task_l_stop_at_lower_bound,

            TASK_L["StpPct_Lower_Bound"],

            np.isclose(
                TASK_M["StpPct_Lower_Bound_Share_Min"],
                1.0,
                rtol=0.0,
                atol=1e-12
            ),

            np.isclose(
                TASK_M["StpPct_Lower_Bound_Share_Max"],
                1.0,
                rtol=0.0,
                atol=1e-12
            ),

            not TASK_L["ChnLen_Lower_Bound"],

            not TASK_L["ChnLen_Upper_Bound"],

            TASK_M["Specifications"] == 6,
        ],
    }
)


print()
print("=" * 100)
print("SECTION 3 VALIDATION")
print("=" * 100)

display(parameter_checks)

assert parameter_checks["Passed"].all()


# ------------------------------------------------------------
# 3.6 Interpretation
# ------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 3 INTERPRETATION")
print("=" * 100)

print(
    "1. ChnLen is not fixed across the AUG analyses: "
    "the rolling Task K optimum varies across windows, "
    "Task L selects a full-sample optimum of 710, and "
    "Task M shows moderate variation across alternative "
    "walk-forward specifications."
)

print(
    "2. StpPct behaves differently. The prescribed lower "
    "grid boundary of 0.005 is selected persistently in "
    "Task K, Task L, and Task M."
)

print(
    "3. This persistent boundary solution should not be "
    "interpreted as precise evidence that 0.005 is the "
    "unconstrained optimal stop percentage."
)

print(
    "4. Because values below 0.005 are outside the prescribed "
    "grid, the experiment cannot determine whether the "
    "objective would continue improving for tighter stops."
)

print(
    "5. The original grid is intentionally retained rather "
    "than expanded ex post, preserving consistency with the "
    "project specification and avoiding a new layer of "
    "data-driven parameter search."
)

print(
    "6. The StpPct boundary solution is therefore treated as "
    "a material parameter-identification limitation in the "
    "final AUG assessment."
)

print()
print("SECTION 3 VALIDATION PASSED.")

TASK N — PARAMETER BEHAVIOR ACROSS TASKS K / L / M


,Evidence,ChnLen Result,StpPct Result,Lower-Bound Evidence
0,Task K — Rolling OOS,Median = 1920; 8 unique values,Median = 0.005,True
1,Task L — Full-Sample IS,Global optimum = 710,Global optimum = 0.005,True
2,Task M — Sensitivity,Median range = 640 to 1920,Lower-bound share = 100% to 100%,True



STOP-PARAMETER BOUNDARY AUDIT
Prescribed StpPct grid : 0.005 to 0.100



,Analysis,Boundary Evidence,At Prescribed Lower Bound
0,Task K — Rolling OOS,100% of rolling windows,True
1,Task L — Full-Sample IS,Global optimum StpPct = 0.005,True
2,Task M — Sensitivity,100% to 100%,True



CHANNEL-LENGTH BEHAVIOR


...


PARAMETER-IDENTIFICATION FLAGS


,Diagnostic,Result
0,Task K stop lower-bound frequency,1.000000
1,Task L stop optimum at lower bound,True
2,Task M stop lower-bound persistence,True
3,Task L ChnLen at lower bound,False
4,Task L ChnLen at upper bound,False
5,Task M tested alternative T/tau specifications,6



SECTION 3 VALIDATION


,Validation Check,Passed
0,Task K StpPct median equals prescribed lower b...,True
1,Task K lower-bound frequency equals 100%,True
2,Task L optimum StpPct equals prescribed lower ...,True
3,Task L lower-bound flag is True,True
4,Task M lower-bound share minimum equals 100%,True
5,Task M lower-bound share maximum equals 100%,True
6,Task L ChnLen is not at lower boundary,True
7,Task L ChnLen is not at upper boundary,True
8,Task M contains six sensitivity specifications,True



SECTION 3 INTERPRETATION
1. ChnLen is not fixed across the AUG analyses: the rolling Task K optimum varies across windows, Task L selects a full-sample optimum of 710, and Task M shows moderate variation across alternative walk-forward specifications.
2. StpPct behaves differently. The prescribed lower grid boundary of 0.005 is selected persistently in Task K, Task L, and Task M.
3. This persistent boundary solution should not be interpreted as precise evidence that 0.005 is the unconstrained optimal stop percentage.
4. Because values below 0.005 are outside the prescribed grid, the experiment cannot determine whether the objective would continue improving for tighter stops.
5. The original grid is intentionally retained rather than expanded ex post, preserving consistency with the project specification and avoiding a new layer of data-driven parameter search.
6. The StpPct boundary solution is therefore treated as a material parameter-identification limitation in the final AUG assess

## 4. OOS Concentration and Session-Gap Diagnostics

The strong headline and trade-level OOS results must be interpreted together with the diagnostic evidence established in Task K.

Two issues are particularly important.

### 4.1 Temporal concentration

All 15 finalized Task K OOS quarters are profitable. However, the distribution of cumulative OOS profit is not uniform across time.

The largest OOS quarter contributes approximately 41.06% of total net OOS profit, while the three largest quarters together contribute approximately 63.11%.

Strong performance therefore exists across all evaluated quarters, but a substantial fraction of the total economic result is concentrated in a relatively small number of particularly profitable periods.

### 4.2 Session-gap dependence

Task K also identified substantial dependence on bars following long inter-session gaps.

Approximately 76.67% of total OOS net P&L is attributed to the identified session-gap bars. Because the provided 5-minute dataset does not observe the price path during those gaps, the backtest cannot determine whether trailing stops, channel reversals, or other intragap events would have occurred before the next observed bar.

This creates a material **data-path limitation**.

Importantly, the Task K non-gap diagnostic remains profitable after removing the P&L attributed to identified session-gap bars. The diagnostic produces a CAGR of approximately 34.00%, although with weaker risk-adjusted performance than the finalized baseline.

The non-gap result does not eliminate the session-gap concern. Instead, it shows that the historical result is not entirely attributable to gap-bar P&L while confirming that the magnitude of the headline AUG performance is materially dependent on the treatment of unobserved inter-session price paths.

These diagnostics are therefore incorporated directly into the final Task N assessment rather than treated as secondary implementation details.

In [5]:
# ============================================================
# 4. OOS Concentration and Session-Gap Diagnostics
# ============================================================


# ------------------------------------------------------------
# 4.1 Finalized Task K diagnostic evidence
# ------------------------------------------------------------

diagnostic_summary = pd.DataFrame(
    {
        "Diagnostic": [
            "Positive OOS windows",
            "Negative OOS windows",
            "Largest-quarter share of net OOS P&L",
            "Top-3-quarter share of net OOS P&L",
            "Session-gap share of net OOS P&L",
            "Non-gap diagnostic CAGR",
            "Non-gap diagnostic MDD",
            "Non-gap diagnostic Calmar",
        ],

        "Result": [
            TASK_K["Positive_OOS_Windows"],
            TASK_K["Negative_OOS_Windows"],
            TASK_K["Largest_Quarter_Share"],
            TASK_K["Top3_Quarter_Share"],
            TASK_K["Session_Gap_PnL_Share"],
            TASK_K["NonGap_Diagnostic_CAGR"],
            TASK_K["NonGap_Diagnostic_MDD_Pct"],
            TASK_K["NonGap_Diagnostic_Calmar"],
        ],
    }
)


print("=" * 100)
print("TASK N — OOS CONCENTRATION AND SESSION-GAP DIAGNOSTICS")
print("=" * 100)

display(diagnostic_summary)


# ------------------------------------------------------------
# 4.2 Temporal concentration
# ------------------------------------------------------------

largest_quarter_share = (
    TASK_K["Largest_Quarter_Share"]
)

top3_quarter_share = (
    TASK_K["Top3_Quarter_Share"]
)

remaining_after_largest = (
    1.0
    -
    largest_quarter_share
)

remaining_after_top3 = (
    1.0
    -
    top3_quarter_share
)


concentration_table = pd.DataFrame(
    {
        "Measure": [
            "Largest quarter",
            "Top 3 quarters",
            "Remaining after largest quarter",
            "Remaining after top 3 quarters",
        ],

        "Share of Total Net OOS P&L": [
            largest_quarter_share,
            top3_quarter_share,
            remaining_after_largest,
            remaining_after_top3,
        ],
    }
)


print()
print("=" * 100)
print("TEMPORAL PROFIT CONCENTRATION")
print("=" * 100)

display(concentration_table)

print(
    f"Profitable OOS windows : "
    f"{TASK_K['Positive_OOS_Windows']} / "
    f"{TASK_K['OOS_Windows']}"
)

print(
    f"Largest-quarter share  : "
    f"{100 * largest_quarter_share:.2f}%"
)

print(
    f"Top-3-quarter share     : "
    f"{100 * top3_quarter_share:.2f}%"
)


# ------------------------------------------------------------
# 4.3 Session-gap attribution
# ------------------------------------------------------------

gap_share = (
    TASK_K["Session_Gap_PnL_Share"]
)

non_gap_share = (
    1.0
    -
    gap_share
)

gap_attribution_table = pd.DataFrame(
    {
        "Component": [
            "Session-gap attributed P&L",
            "Non-gap attributed P&L",
        ],

        "Share of Net OOS P&L": [
            gap_share,
            non_gap_share,
        ],
    }
)


print()
print("=" * 100)
print("SESSION-GAP P&L ATTRIBUTION")
print("=" * 100)

display(gap_attribution_table)

print(
    f"Session-gap P&L share : "
    f"{100 * gap_share:.2f}%"
)

print(
    f"Non-gap P&L share     : "
    f"{100 * non_gap_share:.2f}%"
)


# ------------------------------------------------------------
# 4.4 Baseline vs non-gap diagnostic
# ------------------------------------------------------------

non_gap_comparison = pd.DataFrame(
    {
        "Metric": [
            "CAGR",
            "Maximum Drawdown (%)",
            "Calmar",
        ],

        "Task K Finalized Baseline": [
            TASK_K["CAGR"],
            TASK_K["Max_Drawdown_Pct"],
            TASK_K["Calmar"],
        ],

        "Task K Non-Gap Diagnostic": [
            TASK_K["NonGap_Diagnostic_CAGR"],
            TASK_K["NonGap_Diagnostic_MDD_Pct"],
            TASK_K["NonGap_Diagnostic_Calmar"],
        ],
    }
)


print()
print("=" * 100)
print("BASELINE VS NON-GAP DIAGNOSTIC")
print("=" * 100)

display(non_gap_comparison)


# ------------------------------------------------------------
# 4.5 Diagnostic ratios
# ------------------------------------------------------------

non_gap_cagr_retention = (
    TASK_K["NonGap_Diagnostic_CAGR"]
    /
    TASK_K["CAGR"]
)

non_gap_calmar_retention = (
    TASK_K["NonGap_Diagnostic_Calmar"]
    /
    TASK_K["Calmar"]
)

non_gap_abs_mdd_ratio = (
    abs(TASK_K["NonGap_Diagnostic_MDD_Pct"])
    /
    abs(TASK_K["Max_Drawdown_Pct"])
)


print()
print("=" * 100)
print("NON-GAP DIAGNOSTIC RETENTION")
print("=" * 100)

print(
    f"CAGR retention             : "
    f"{non_gap_cagr_retention:.4f}"
)

print(
    f"Calmar retention           : "
    f"{non_gap_calmar_retention:.4f}"
)

print(
    f"Absolute MDD ratio         : "
    f"{non_gap_abs_mdd_ratio:.4f}"
)


# ------------------------------------------------------------
# 4.6 Validation
# ------------------------------------------------------------

diagnostic_checks = pd.DataFrame(
    {
        "Validation Check": [
            "All 15 finalized OOS windows are positive",
            "No finalized OOS windows are negative",
            "Largest-quarter share is between 0 and 1",
            "Top-3-quarter share is between 0 and 1",
            "Top-3 share is at least largest-quarter share",
            "Session-gap P&L share is between 0 and 1",
            "Gap and non-gap shares reconcile to 100%",
            "Non-gap diagnostic CAGR remains positive",
            "Non-gap diagnostic Calmar remains positive",
            "Non-gap diagnostic has deeper percentage MDD than baseline",
        ],

        "Passed": [
            (
                TASK_K["Positive_OOS_Windows"]
                ==
                TASK_K["OOS_Windows"]
                ==
                15
            ),

            TASK_K["Negative_OOS_Windows"] == 0,

            (
                0.0
                <= largest_quarter_share
                <= 1.0
            ),

            (
                0.0
                <= top3_quarter_share
                <= 1.0
            ),

            (
                top3_quarter_share
                >=
                largest_quarter_share
            ),

            (
                0.0
                <= gap_share
                <= 1.0
            ),

            np.isclose(
                gap_share + non_gap_share,
                1.0,
                rtol=0.0,
                atol=1e-12
            ),

            TASK_K["NonGap_Diagnostic_CAGR"] > 0.0,

            TASK_K["NonGap_Diagnostic_Calmar"] > 0.0,

            (
                abs(
                    TASK_K["NonGap_Diagnostic_MDD_Pct"]
                )
                >
                abs(
                    TASK_K["Max_Drawdown_Pct"]
                )
            ),
        ],
    }
)


print()
print("=" * 100)
print("SECTION 4 VALIDATION")
print("=" * 100)

display(diagnostic_checks)

assert diagnostic_checks["Passed"].all()


# ------------------------------------------------------------
# 4.7 Interpretation
# ------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 4 INTERPRETATION")
print("=" * 100)

print(
    "1. All 15 finalized Task K OOS windows are profitable, "
    "so the positive historical OOS result is not confined "
    "to a single profitable quarter."
)

print(
    "2. Profit magnitude is nevertheless concentrated: the "
    "largest quarter contributes about 41% of total net OOS "
    "P&L and the three largest quarters contribute about 63%."
)

print(
    "3. Approximately 76.67% of finalized net OOS P&L is "
    "attributed to identified session-gap bars, making "
    "session-gap treatment a material data-path limitation."
)

print(
    "4. Because the provided 5-minute data do not observe "
    "the intragap price path, the backtest cannot determine "
    "whether stops or reversals would have occurred during "
    "those unobserved intervals."
)

print(
    "5. The non-gap diagnostic remains historically profitable, "
    "with positive CAGR and Calmar, showing that the AUG result "
    "is not entirely attributable to session-gap P&L."
)

print(
    "6. However, non-gap performance is materially weaker and "
    "its percentage drawdown is deeper, so the magnitude of the "
    "headline AUG result remains substantially dependent on "
    "session-gap attribution."
)

print()
print("SECTION 4 VALIDATION PASSED.")

TASK N — OOS CONCENTRATION AND SESSION-GAP DIAGNOSTICS


,Diagnostic,Result
0,Positive OOS windows,15.000000
1,Negative OOS windows,0.000000
2,Largest-quarter share of net OOS P&L,0.410604
3,Top-3-quarter share of net OOS P&L,0.631136
4,Session-gap share of net OOS P&L,0.766696
5,Non-gap diagnostic CAGR,0.339955
6,Non-gap diagnostic MDD,-0.125938
7,Non-gap diagnostic Calmar,2.699387



TEMPORAL PROFIT CONCENTRATION


,Measure,Share of Total Net OOS P&L
0,Largest quarter,0.410604
1,Top 3 quarters,0.631136
2,Remaining after largest quarter,0.589396
3,Remaining after top 3 quarters,0.368864


Profitable OOS windows : 15 / 15
Largest-quarter share  : 41.06%
Top-3-quarter share     : 63.11%

SESSION-GAP P&L ATTRIBUTION


...

Session-gap P&L share : 76.67%
Non-gap P&L share     : 23.33%

BASELINE VS NON-GAP DIAGNOSTIC


,Metric,Task K Finalized Baseline,Task K Non-Gap Diagnostic
0,CAGR,0.825766,0.339955
1,Maximum Drawdown (%),-0.099059,-0.125938
2,Calmar,8.336143,2.699387



NON-GAP DIAGNOSTIC RETENTION
CAGR retention             : 0.4117
Calmar retention           : 0.3238
Absolute MDD ratio         : 1.2713

SECTION 4 VALIDATION


,Validation Check,Passed
0,All 15 finalized OOS windows are positive,True
1,No finalized OOS windows are negative,True
2,Largest-quarter share is between 0 and 1,True
3,Top-3-quarter share is between 0 and 1,True
4,Top-3 share is at least largest-quarter share,True
5,Session-gap P&L share is between 0 and 1,True
6,Gap and non-gap shares reconcile to 100%,True
7,Non-gap diagnostic CAGR remains positive,True
8,Non-gap diagnostic Calmar remains positive,True
9,Non-gap diagnostic has deeper percentage MDD t...,True



SECTION 4 INTERPRETATION
1. All 15 finalized Task K OOS windows are profitable, so the positive historical OOS result is not confined to a single profitable quarter.
2. Profit magnitude is nevertheless concentrated: the largest quarter contributes about 41% of total net OOS P&L and the three largest quarters contribute about 63%.
3. Approximately 76.67% of finalized net OOS P&L is attributed to identified session-gap bars, making session-gap treatment a material data-path limitation.
4. Because the provided 5-minute data do not observe the intragap price path, the backtest cannot determine whether stops or reversals would have occurred during those unobserved intervals.
5. The non-gap diagnostic remains historically profitable, with positive CAGR and Calmar, showing that the AUG result is not entirely attributable to session-gap P&L.
6. However, non-gap performance is materially weaker and its percentage drawdown is deeper, so the magnitude of the headline AUG result remains substanti

## 5. Integration of Walk-Forward Sensitivity Evidence

The Task K versus Task L comparison shows that the finalized rolling OOS result does not exhibit a clear collapse in risk-adjusted or trade-level performance relative to the full-sample hindsight benchmark.

However, that comparison alone does not establish whether the strong Task K result is unusually dependent on its specific walk-forward design.

Task M addresses this issue by varying only the in-sample estimation window \(T\) and OOS evaluation horizon \(\tau\), while preserving the finalized strategy engine, transaction-cost assumptions, objective function, and full professor-specified parameter grid.

The six tested specifications are:

\[
T \in \{4,5,6\}\text{ years},
\qquad
\tau \in \{3,6\}\text{ months}.
\]

For a fair comparison, Task M evaluates all six specifications over their identical common OOS period from July 2024 through December 2025.

Across this common period:

- all six specifications remain profitable;
- CAGR ranges from approximately 167.68% to 183.05%;
- Daily Sharpe ranges from approximately 3.98 to 4.30;
- maximum percentage drawdown is approximately -14.14% across all six specifications;
- Calmar ranges from approximately 11.86 to 12.94.

These results provide evidence that the strong historical AUG OOS result is not unique to the original 4-year / 3-month walk-forward specification within the tested range.

This evidence should nevertheless be interpreted cautiously. The common comparison period is relatively short, the longest-window specifications contain only a small number of OOS windows, and the stop-loss parameter remains at the prescribed lower grid boundary throughout the sensitivity analysis.

Task M therefore strengthens the historical robustness evidence, but does not eliminate the parameter-boundary, sample-length, temporal-concentration, or session-gap limitations identified elsewhere in the AUG analysis.

In [6]:
# ------------------------------------------------------------
# 5.1 Finalized Task M sensitivity summary
# ------------------------------------------------------------

sensitivity_summary = pd.DataFrame(
    {
        "Item": [
            "Sensitivity specifications",
            "T values (years)",
            "Tau values (months)",
            "Common OOS start",
            "Common OOS end",
            "Common OOS bars",
            "Common-period CAGR minimum",
            "Common-period CAGR maximum",
            "Common-period Sharpe minimum",
            "Common-period Sharpe maximum",
            "Common-period MDD minimum",
            "Common-period MDD maximum",
            "Common-period Calmar minimum",
            "Common-period Calmar maximum",
            "Minimum OOS windows",
            "Maximum OOS windows",
        ],

        "Result": [
            TASK_M["Specifications"],
            str(TASK_M["T_Years"]),
            str(TASK_M["Tau_Months"]),
            TASK_M["Common_Start"],
            TASK_M["Common_End"],
            TASK_M["Common_Bars"],
            TASK_M["CAGR_Min"],
            TASK_M["CAGR_Max"],
            TASK_M["Sharpe_Min"],
            TASK_M["Sharpe_Max"],
            TASK_M["MDD_Pct_Min"],
            TASK_M["MDD_Pct_Max"],
            TASK_M["Calmar_Min"],
            TASK_M["Calmar_Max"],
            TASK_M["Minimum_OOS_Windows"],
            TASK_M["Maximum_OOS_Windows"],
        ],
    }
)


print("=" * 100)
print("TASK N — TASK M SENSITIVITY EVIDENCE")
print("=" * 100)

display(sensitivity_summary)


# ------------------------------------------------------------
# 5.2 Performance-range widths
# ------------------------------------------------------------

cagr_range_width = (
    TASK_M["CAGR_Max"]
    -
    TASK_M["CAGR_Min"]
)

sharpe_range_width = (
    TASK_M["Sharpe_Max"]
    -
    TASK_M["Sharpe_Min"]
)

mdd_range_width = (
    TASK_M["MDD_Pct_Max"]
    -
    TASK_M["MDD_Pct_Min"]
)

calmar_range_width = (
    TASK_M["Calmar_Max"]
    -
    TASK_M["Calmar_Min"]
)


range_summary = pd.DataFrame(
    {
        "Metric": [
            "CAGR",
            "Daily Sharpe",
            "Maximum Drawdown (%)",
            "Calmar",
        ],

        "Minimum": [
            TASK_M["CAGR_Min"],
            TASK_M["Sharpe_Min"],
            TASK_M["MDD_Pct_Min"],
            TASK_M["Calmar_Min"],
        ],

        "Maximum": [
            TASK_M["CAGR_Max"],
            TASK_M["Sharpe_Max"],
            TASK_M["MDD_Pct_Max"],
            TASK_M["Calmar_Max"],
        ],

        "Range Width": [
            cagr_range_width,
            sharpe_range_width,
            mdd_range_width,
            calmar_range_width,
        ],
    }
)


print()
print("=" * 100)
print("COMMON-PERIOD PERFORMANCE RANGES")
print("=" * 100)

display(range_summary)


# ------------------------------------------------------------
# 5.3 Parameter sensitivity evidence
# ------------------------------------------------------------

sensitivity_parameter_summary = pd.DataFrame(
    {
        "Parameter Diagnostic": [
            "Median ChnLen minimum",
            "Median ChnLen maximum",
            "StpPct lower-bound share minimum",
            "StpPct lower-bound share maximum",
        ],

        "Result": [
            TASK_M["Median_ChnLen_Min"],
            TASK_M["Median_ChnLen_Max"],
            TASK_M["StpPct_Lower_Bound_Share_Min"],
            TASK_M["StpPct_Lower_Bound_Share_Max"],
        ],
    }
)


print()
print("=" * 100)
print("TASK M PARAMETER SENSITIVITY")
print("=" * 100)

display(sensitivity_parameter_summary)


# ------------------------------------------------------------
# 5.4 Robustness flags
# ------------------------------------------------------------

all_common_profitable = (
    TASK_M["CAGR_Min"] > 0.0
)

all_common_sharpe_positive = (
    TASK_M["Sharpe_Min"] > 0.0
)

all_common_calmar_positive = (
    TASK_M["Calmar_Min"] > 0.0
)

mdd_consistent = np.isclose(
    TASK_M["MDD_Pct_Min"],
    TASK_M["MDD_Pct_Max"],
    rtol=0.0,
    atol=1e-12
)

stop_boundary_persistent = (
    np.isclose(
        TASK_M["StpPct_Lower_Bound_Share_Min"],
        1.0,
        rtol=0.0,
        atol=1e-12
    )
    and
    np.isclose(
        TASK_M["StpPct_Lower_Bound_Share_Max"],
        1.0,
        rtol=0.0,
        atol=1e-12
    )
)


robustness_flags = pd.DataFrame(
    {
        "Diagnostic": [
            "All tested specifications profitable on common period",
            "All tested specifications have positive Sharpe",
            "All tested specifications have positive Calmar",
            "Common-period MDD identical across specifications",
            "StpPct lower-bound solution persists",
            "Longest specifications have limited OOS windows",
        ],

        "Result": [
            all_common_profitable,
            all_common_sharpe_positive,
            all_common_calmar_positive,
            mdd_consistent,
            stop_boundary_persistent,
            TASK_M["Minimum_OOS_Windows"],
        ],
    }
)


print()
print("=" * 100)
print("ROBUSTNESS AND LIMITATION FLAGS")
print("=" * 100)

display(robustness_flags)


# ------------------------------------------------------------
# 5.5 Validation
# ------------------------------------------------------------

sensitivity_checks = pd.DataFrame(
    {
        "Validation Check": [
            "Six Task M specifications included",
            "T sensitivity values are 4, 5, and 6 years",
            "Tau sensitivity values are 3 and 6 months",
            "Common OOS period contains positive number of bars",
            "Common-period minimum CAGR is positive",
            "Common-period minimum Sharpe is positive",
            "Common-period minimum Calmar is positive",
            "Common-period MDD values reconcile",
            "StpPct lower-bound persistence retained",
            "Minimum OOS-window count is positive",
        ],

        "Passed": [
            TASK_M["Specifications"] == 6,

            tuple(TASK_M["T_Years"]) == (4, 5, 6),

            tuple(TASK_M["Tau_Months"]) == (3, 6),

            TASK_M["Common_Bars"] > 0,

            all_common_profitable,

            all_common_sharpe_positive,

            all_common_calmar_positive,

            mdd_consistent,

            stop_boundary_persistent,

            TASK_M["Minimum_OOS_Windows"] > 0,
        ],
    }
)


print()
print("=" * 100)
print("SECTION 5 VALIDATION")
print("=" * 100)

display(sensitivity_checks)

assert sensitivity_checks["Passed"].all()


# ------------------------------------------------------------
# 5.6 Interpretation
# ------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 5 INTERPRETATION")
print("=" * 100)

print(
    "1. All six Task M walk-forward specifications remain "
    "profitable over the identical common OOS period."
)

print(
    "2. Risk-adjusted performance remains strong across the "
    "tested T/tau choices, with positive Sharpe and Calmar "
    "ratios under every specification."
)

print(
    "3. The strong historical AUG OOS result is therefore not "
    "unique to the original 4-year IS / 3-month OOS window "
    "within the tested sensitivity range."
)

print(
    "4. ChnLen shows moderate sensitivity to the walk-forward "
    "design, while StpPct remains persistently located at the "
    "0.005 lower grid boundary."
)

print(
    "5. The sensitivity evidence is limited by the relatively "
    "short common comparison period and by the small number of "
    "OOS windows available for the longest specifications."
)

print(
    "6. Task M strengthens the historical robustness evidence "
    "but does not eliminate the parameter-boundary, "
    "session-gap, temporal-concentration, or sample-length "
    "limitations."
)

print()
print("SECTION 5 VALIDATION PASSED.")

TASK N — TASK M SENSITIVITY EVIDENCE


,Item,Result
0,Sensitivity specifications,6
1,T values (years),"(4, 5, 6)"
2,Tau values (months),"(3, 6)"
3,Common OOS start,2024-07-01 09:05:00
4,Common OOS end,2025-12-31 15:00:00
5,Common OOS bars,26496
6,Common-period CAGR minimum,1.676813
7,Common-period CAGR maximum,1.830535
8,Common-period Sharpe minimum,3.979571
9,Common-period Sharpe maximum,4.301115



COMMON-PERIOD PERFORMANCE RANGES


,Metric,Minimum,Maximum,Range Width
0,CAGR,1.676813,1.830535,0.153722
1,Daily Sharpe,3.979571,4.301115,0.321544
2,Maximum Drawdown (%),-0.141416,-0.141416,0.000000
3,Calmar,11.857294,12.944313,1.087019



TASK M PARAMETER SENSITIVITY


...


ROBUSTNESS AND LIMITATION FLAGS


,Diagnostic,Result
0,All tested specifications profitable on common...,True
1,All tested specifications have positive Sharpe,True
2,All tested specifications have positive Calmar,True
3,Common-period MDD identical across specifications,True
4,StpPct lower-bound solution persists,True
5,Longest specifications have limited OOS windows,3



SECTION 5 VALIDATION


,Validation Check,Passed
0,Six Task M specifications included,True
1,"T sensitivity values are 4, 5, and 6 years",True
2,Tau sensitivity values are 3 and 6 months,True
3,Common OOS period contains positive number of ...,True
4,Common-period minimum CAGR is positive,True
5,Common-period minimum Sharpe is positive,True
6,Common-period minimum Calmar is positive,True
7,Common-period MDD values reconcile,True
8,StpPct lower-bound persistence retained,True
9,Minimum OOS-window count is positive,True



SECTION 5 INTERPRETATION
1. All six Task M walk-forward specifications remain profitable over the identical common OOS period.
2. Risk-adjusted performance remains strong across the tested T/tau choices, with positive Sharpe and Calmar ratios under every specification.
3. The strong historical AUG OOS result is therefore not unique to the original 4-year IS / 3-month OOS window within the tested sensitivity range.
4. ChnLen shows moderate sensitivity to the walk-forward design, while StpPct remains persistently located at the 0.005 lower grid boundary.
5. The sensitivity evidence is limited by the relatively short common comparison period and by the small number of OOS windows available for the longest specifications.
6. Task M strengthens the historical robustness evidence but does not eliminate the parameter-boundary, session-gap, temporal-concentration, or sample-length limitations.

SECTION 5 VALIDATION PASSED.


## 6. Final AUG Assessment

Tasks K–N provide a sequence of complementary tests for the AUG / SHFE Gold Futures replication.

### Historical OOS evidence

The finalized Task K rolling walk-forward evaluation produces strong historical OOS performance. Across 15 non-overlapping OOS windows, all 15 windows are profitable. Risk-adjusted performance remains strong, with a Daily Sharpe ratio of approximately 3.61 and a Calmar ratio of approximately 8.34.

### Comparison with the hindsight benchmark

Task L provides a full-sample hindsight IS benchmark rather than a directly competing OOS strategy.

Relative to this benchmark, the Task K OOS result does not exhibit a clear collapse in normalized performance or trade-level quality. OOS Sharpe, profit factor, and win rate remain broadly comparable to their hindsight-IS values, while expectancy and payoff ratio remain strong.

However, Task K experiences a deeper percentage drawdown than the Task L benchmark. Because the two evaluations cover different periods and use different evaluation designs, these comparisons are descriptive and should not be interpreted as evidence that OOS performance is superior to IS performance.

### Walk-forward robustness

Task M shows that the strong historical OOS result is not unique to the original 4-year IS / 3-month OOS design within the tested range.

Across six alternative \(T/\tau\) specifications evaluated over an identical common OOS period, all specifications remain profitable and retain strong risk-adjusted performance.

This strengthens the historical robustness evidence with respect to reasonable changes in the walk-forward window design.

### Material limitations

The AUG evidence nevertheless contains several important limitations.

First, `StpPct = 0.005` is selected at the prescribed lower grid boundary throughout Task K, at the Task L full-sample optimum, and throughout the Task M sensitivity analysis. The experiment therefore does not identify an unconstrained interior optimum for the stop parameter.

Second, OOS profit magnitude is temporally concentrated. Although all 15 Task K OOS windows are profitable, the largest quarter contributes approximately 41% of total net OOS P&L and the three largest quarters contribute approximately 63%.

Third, approximately 76.67% of finalized Task K net OOS P&L is attributed to identified session-gap bars. The provided 5-minute data do not reveal the intragap price path, so the backtest cannot determine whether stops or reversals would have occurred during those unobserved intervals.

The non-gap diagnostic remains historically profitable, which indicates that the AUG result is not entirely attributable to session-gap P&L. However, its performance is materially weaker than the finalized baseline.

Finally, the available AUG history is relatively short, and the longest Task M sensitivity specifications contain only a small number of OOS windows.

### Overall assessment

Taken together, the AUG replication provides evidence of strong historical OOS performance under the prescribed Channel WithDDControl framework. The trade-level edge does not clearly collapse relative to the full-sample hindsight benchmark, and the result remains strong across the tested walk-forward window specifications.

However, the magnitude of the headline result is materially exposed to session-gap attribution, the stop parameter remains a persistent lower-bound solution, profit is temporally concentrated, and the available OOS history is limited.

The appropriate conclusion is therefore one of **historical robustness within the provided bar data**, rather than proof of deployable live performance, parameter invariance, or absence of overfitting and data-path bias.

In [7]:
# ------------------------------------------------------------
# 6.1 Consolidated evidence
# ------------------------------------------------------------

final_evidence = pd.DataFrame(
    {
        "Dimension": [
            "Rolling OOS performance",
            "OOS window consistency",
            "Sharpe vs hindsight IS",
            "Profit factor vs hindsight IS",
            "Trade expectancy",
            "Walk-forward sensitivity",
            "Channel-length behavior",
            "Stop-parameter identification",
            "Temporal concentration",
            "Session-gap dependence",
            "Non-gap diagnostic",
            "Sample-length limitation",
        ],

        "Evidence": [
            (
                f"CAGR {100 * TASK_K['CAGR']:.2f}%, "
                f"Sharpe {TASK_K['Daily_Sharpe']:.4f}, "
                f"Calmar {TASK_K['Calmar']:.4f}"
            ),

            (
                f"{TASK_K['Positive_OOS_Windows']} / "
                f"{TASK_K['OOS_Windows']} "
                f"OOS windows profitable"
            ),

            (
                f"{TASK_K['Daily_Sharpe']:.4f} OOS vs "
                f"{TASK_L['Daily_Sharpe']:.4f} hindsight IS"
            ),

            (
                f"{TASK_K['Profit_Factor']:.4f} OOS vs "
                f"{TASK_L['Profit_Factor']:.4f} hindsight IS"
            ),

            (
                f"{TASK_K['Expectancy']:,.2f} CNY/trade OOS vs "
                f"{TASK_L['Expectancy']:,.2f} CNY/trade hindsight IS"
            ),

            (
                f"{TASK_M['Specifications']} specifications; "
                f"common-period CAGR "
                f"{100 * TASK_M['CAGR_Min']:.2f}%–"
                f"{100 * TASK_M['CAGR_Max']:.2f}%"
            ),

            (
                f"Task K median {TASK_K['Median_ChnLen']:.0f}; "
                f"Task M median range "
                f"{TASK_M['Median_ChnLen_Min']:.0f}–"
                f"{TASK_M['Median_ChnLen_Max']:.0f}"
            ),

            (
                "StpPct = 0.005 lower-bound solution "
                "persists across Tasks K / L / M"
            ),

            (
                f"Largest quarter = "
                f"{100 * TASK_K['Largest_Quarter_Share']:.2f}% "
                f"of net OOS P&L; top 3 = "
                f"{100 * TASK_K['Top3_Quarter_Share']:.2f}%"
            ),

            (
                f"{100 * TASK_K['Session_Gap_PnL_Share']:.2f}% "
                f"of net OOS P&L attributed to "
                f"identified session-gap bars"
            ),

            (
                f"CAGR "
                f"{100 * TASK_K['NonGap_Diagnostic_CAGR']:.2f}%, "
                f"Calmar "
                f"{TASK_K['NonGap_Diagnostic_Calmar']:.4f}"
            ),

            (
                f"Task M longest specifications contain as few as "
                f"{TASK_M['Minimum_OOS_Windows']} OOS windows"
            ),
        ],

        "Assessment": [
            "Strong historical OOS evidence",
            "Positive across all finalized OOS windows",
            "No clear Sharpe collapse",
            "No clear profit-factor collapse",
            "Positive trade-level edge retained",
            "Limited sensitivity within tested T/tau range",
            "Moderate parameter variation",
            "Material boundary limitation",
            "Material concentration limitation",
            "Material data-path limitation",
            "Positive but materially weaker",
            "Material inference limitation",
        ],
    }
)


print("=" * 110)
print("TASK N — CONSOLIDATED AUG EVIDENCE")
print("=" * 110)

display(final_evidence)


# ------------------------------------------------------------
# 6.2 Quantitative final-summary table
# ------------------------------------------------------------

final_summary = pd.DataFrame(
    {
        "Item": [
            "Task K OOS Start",
            "Task K OOS End",
            "Task K OOS Windows",
            "Task K Positive OOS Windows",
            "Task K CAGR",
            "Task K Daily Sharpe",
            "Task K Maximum Drawdown",
            "Task K Calmar",
            "Task K Profit Factor",
            "Task K Expectancy",
            "Task L Hindsight-IS CAGR",
            "Task L Hindsight-IS Daily Sharpe",
            "Task L Hindsight-IS Maximum Drawdown",
            "Task L Hindsight-IS Calmar",
            "Task L Hindsight-IS Profit Factor",
            "Task M Specifications",
            "Task M Common-Period CAGR Range",
            "Task M Common-Period Sharpe Range",
            "Task M Common-Period Calmar Range",
            "Task K Largest-Quarter P&L Share",
            "Task K Top-3-Quarter P&L Share",
            "Task K Session-Gap P&L Share",
            "Task K Non-Gap Diagnostic CAGR",
            "Task K Non-Gap Diagnostic Calmar",
            "StpPct Boundary Status",
        ],

        "Result": [
            TASK_K["Sample_Start"],
            TASK_K["Sample_End"],
            TASK_K["OOS_Windows"],
            TASK_K["Positive_OOS_Windows"],
            f"{100 * TASK_K['CAGR']:.2f}%",
            f"{TASK_K['Daily_Sharpe']:.4f}",
            f"{100 * TASK_K['Max_Drawdown_Pct']:.2f}%",
            f"{TASK_K['Calmar']:.4f}",
            f"{TASK_K['Profit_Factor']:.4f}",
            f"{TASK_K['Expectancy']:,.2f} CNY/trade",
            f"{100 * TASK_L['CAGR']:.2f}%",
            f"{TASK_L['Daily_Sharpe']:.4f}",
            f"{100 * TASK_L['Max_Drawdown_Pct']:.2f}%",
            f"{TASK_L['Calmar']:.4f}",
            f"{TASK_L['Profit_Factor']:.4f}",
            TASK_M["Specifications"],
            (
                f"{100 * TASK_M['CAGR_Min']:.2f}%–"
                f"{100 * TASK_M['CAGR_Max']:.2f}%"
            ),
            (
                f"{TASK_M['Sharpe_Min']:.4f}–"
                f"{TASK_M['Sharpe_Max']:.4f}"
            ),
            (
                f"{TASK_M['Calmar_Min']:.4f}–"
                f"{TASK_M['Calmar_Max']:.4f}"
            ),
            (
                f"{100 * TASK_K['Largest_Quarter_Share']:.2f}%"
            ),
            (
                f"{100 * TASK_K['Top3_Quarter_Share']:.2f}%"
            ),
            (
                f"{100 * TASK_K['Session_Gap_PnL_Share']:.2f}%"
            ),
            (
                f"{100 * TASK_K['NonGap_Diagnostic_CAGR']:.2f}%"
            ),
            (
                f"{TASK_K['NonGap_Diagnostic_Calmar']:.4f}"
            ),
            (
                "Persistent 0.005 prescribed lower-bound solution"
            ),
        ],
    }
)


print()
print("=" * 110)
print("TASK N — FINAL QUANTITATIVE SUMMARY")
print("=" * 110)

display(final_summary)


# ------------------------------------------------------------
# 6.3 Final evidence flags
# ------------------------------------------------------------

no_sharpe_collapse = (
    TASK_K["Daily_Sharpe"]
    /
    TASK_L["Daily_Sharpe"]
    >
    0.90
)

no_pf_collapse = (
    TASK_K["Profit_Factor"]
    /
    TASK_L["Profit_Factor"]
    >
    0.90
)

all_oos_windows_positive = (
    TASK_K["Positive_OOS_Windows"]
    ==
    TASK_K["OOS_Windows"]
)

sensitivity_profitable = (
    TASK_M["CAGR_Min"] > 0.0
)

sensitivity_sharpe_positive = (
    TASK_M["Sharpe_Min"] > 0.0
)

stop_boundary_limitation = (
    np.isclose(
        TASK_K["StpPct_Lower_Bound_Share"],
        1.0,
        rtol=0.0,
        atol=1e-12
    )
    and
    TASK_L["StpPct_Lower_Bound"]
    and
    np.isclose(
        TASK_M["StpPct_Lower_Bound_Share_Min"],
        1.0,
        rtol=0.0,
        atol=1e-12
    )
    and
    np.isclose(
        TASK_M["StpPct_Lower_Bound_Share_Max"],
        1.0,
        rtol=0.0,
        atol=1e-12
    )
)

temporal_concentration_present = (
    TASK_K["Largest_Quarter_Share"] > 0.25
    and
    TASK_K["Top3_Quarter_Share"] > 0.50
)

session_gap_limitation_present = (
    TASK_K["Session_Gap_PnL_Share"] > 0.50
)

non_gap_result_positive = (
    TASK_K["NonGap_Diagnostic_CAGR"] > 0.0
    and
    TASK_K["NonGap_Diagnostic_Calmar"] > 0.0
)

limited_long_window_evidence = (
    TASK_M["Minimum_OOS_Windows"] <= 3
)


evidence_flags = pd.DataFrame(
    {
        "Evidence Flag": [
            "No severe Sharpe deterioration vs hindsight IS",
            "No severe profit-factor deterioration vs hindsight IS",
            "All finalized Task K OOS windows profitable",
            "All tested Task M specifications profitable on common period",
            "All tested Task M specifications retain positive Sharpe",
            "Persistent StpPct lower-bound limitation identified",
            "Material temporal concentration identified",
            "Material session-gap dependence identified",
            "Non-gap diagnostic remains historically profitable",
            "Longest sensitivity specifications have limited windows",
        ],

        "Result": [
            no_sharpe_collapse,
            no_pf_collapse,
            all_oos_windows_positive,
            sensitivity_profitable,
            sensitivity_sharpe_positive,
            stop_boundary_limitation,
            temporal_concentration_present,
            session_gap_limitation_present,
            non_gap_result_positive,
            limited_long_window_evidence,
        ],
    }
)


print()
print("=" * 110)
print("TASK N — FINAL EVIDENCE FLAGS")
print("=" * 110)

display(evidence_flags)


# ------------------------------------------------------------
# 6.4 Final validation suite
# ------------------------------------------------------------

final_checks = pd.DataFrame(
    {
        "Validation Check": [
            "Task K identified as rolling OOS",
            "Task L identified as hindsight IS benchmark",
            "Task K equity accounting reconciles",
            "Task L equity accounting reconciles",
            "Task K contains 15 OOS windows",
            "All Task K OOS windows are profitable",
            "Task K Sharpe remains above 90% of Task L benchmark",
            "Task K profit factor remains above 90% of Task L benchmark",
            "Task M contains six sensitivity specifications",
            "Task M common-period performance remains profitable",
            "Task M common-period Sharpe remains positive",
            "Persistent StpPct lower-bound solution identified",
            "Temporal concentration limitation identified",
            "Session-gap limitation identified",
            "Non-gap diagnostic remains profitable",
            "Short-history / limited-window limitation identified",
        ],

        "Passed": [
            TASK_K["Evaluation_Type"] == "Rolling OOS",

            (
                TASK_L["Evaluation_Type"]
                ==
                "Full-Sample IS / Hindsight"
            ),

            np.isclose(
                TASK_K["Ending_Equity"]
                -
                TASK_K["Starting_Equity"],
                TASK_K["Net_Profit"],
                rtol=0.0,
                atol=1e-6
            ),

            np.isclose(
                TASK_L["Ending_Equity"]
                -
                TASK_L["Starting_Equity"],
                TASK_L["Net_Profit"],
                rtol=0.0,
                atol=1e-6
            ),

            TASK_K["OOS_Windows"] == 15,

            all_oos_windows_positive,

            no_sharpe_collapse,

            no_pf_collapse,

            TASK_M["Specifications"] == 6,

            sensitivity_profitable,

            sensitivity_sharpe_positive,

            stop_boundary_limitation,

            temporal_concentration_present,

            session_gap_limitation_present,

            non_gap_result_positive,

            limited_long_window_evidence,
        ],
    }
)


print()
print("=" * 110)
print("TASK N — FINAL VALIDATION")
print("=" * 110)

display(final_checks)

assert final_checks["Passed"].all()


# ------------------------------------------------------------
# 6.5 Final interpretation
# ------------------------------------------------------------

print()
print("=" * 110)
print("FINAL AUG ASSESSMENT")
print("=" * 110)

print(
    "1. The finalized AUG replication exhibits strong historical "
    "rolling OOS performance under the prescribed "
    "Channel WithDDControl framework."
)

print(
    "2. Relative to the full-sample hindsight IS benchmark, "
    "OOS Sharpe, profit factor, win rate, expectancy, and "
    "payoff characteristics do not show a clear collapse in "
    "historical trade-level or risk-adjusted performance."
)

print(
    "3. Task M shows that the strong historical OOS result is "
    "not unique to the original 4-year IS / 3-month OOS design "
    "within the tested T/tau range."
)

print(
    "4. The evidence nevertheless contains material limitations: "
    "StpPct is persistently selected at the prescribed lower "
    "boundary, OOS profit is temporally concentrated, and a "
    "large share of net OOS P&L is attributed to session-gap bars."
)

print(
    "5. The non-gap diagnostic remains historically profitable, "
    "but its materially weaker performance confirms that the "
    "magnitude of the headline AUG result is sensitive to "
    "session-gap attribution."
)

print(
    "6. The relatively short AUG history and the limited number "
    "of OOS windows for the longest sensitivity specifications "
    "further constrain statistical inference."
)

print(
    "7. Overall, the AUG evidence supports historical robustness "
    "within the provided bar data, but does not establish "
    "deployable live performance, parameter invariance, or prove "
    "the absence of overfitting or data-path bias."
)

print()
print("ALL TASK N FINAL VALIDATION CHECKS PASSED.")

TASK N — CONSOLIDATED AUG EVIDENCE


,Dimension,Evidence,Assessment
0,Rolling OOS performance,"CAGR 82.58%, Sharpe 3.6086, Calmar 8.3361",Strong historical OOS evidence
1,OOS window consistency,15 / 15 OOS windows profitable,Positive across all finalized OOS windows
2,Sharpe vs hindsight IS,3.6086 OOS vs 3.6873 hindsight IS,No clear Sharpe collapse
3,Profit factor vs hindsight IS,7.1861 OOS vs 7.4152 hindsight IS,No clear profit-factor collapse
4,Trade expectancy,"4,684.54 CNY/trade OOS vs 3,393.51 CNY/trade h...",Positive trade-level edge retained
5,Walk-forward sensitivity,6 specifications; common-period CAGR 167.68%–1...,Limited sensitivity within tested T/tau range
6,Channel-length behavior,Task K median 1920; Task M median range 640–1920,Moderate parameter variation
7,Stop-parameter identification,StpPct = 0.005 lower-bound solution persists a...,Material boundary limitation
8,Temporal concentration,Largest quarter = 41.06% of net OOS P&L; top 3...,Material concentration limitation
9,Session-gap dependence,76.67% of net OOS P&L attributed to identified...,Material data-path limitation



TASK N — FINAL QUANTITATIVE SUMMARY


,Item,Result
0,Task K OOS Start,2022-07-01 09:05:00
1,Task K OOS End,2026-03-31 15:00:00
2,Task K OOS Windows,15
3,Task K Positive OOS Windows,15
4,Task K CAGR,82.58%
5,Task K Daily Sharpe,3.6086
6,Task K Maximum Drawdown,-9.91%
7,Task K Calmar,8.3361
8,Task K Profit Factor,7.1861
9,Task K Expectancy,"4,684.54 CNY/trade"



TASK N — FINAL EVIDENCE FLAGS


,Evidence Flag,Result
0,No severe Sharpe deterioration vs hindsight IS,True
1,No severe profit-factor deterioration vs hinds...,True
2,All finalized Task K OOS windows profitable,True
3,All tested Task M specifications profitable on...,True
4,All tested Task M specifications retain positi...,True
5,Persistent StpPct lower-bound limitation ident...,True
6,Material temporal concentration identified,True
7,Material session-gap dependence identified,True
8,Non-gap diagnostic remains historically profit...,True
9,Longest sensitivity specifications have limite...,True



TASK N — FINAL VALIDATION


,Validation Check,Passed
0,Task K identified as rolling OOS,True
1,Task L identified as hindsight IS benchmark,True
2,Task K equity accounting reconciles,True
3,Task L equity accounting reconciles,True
4,Task K contains 15 OOS windows,True
5,All Task K OOS windows are profitable,True
6,Task K Sharpe remains above 90% of Task L benc...,True
7,Task K profit factor remains above 90% of Task...,True
8,Task M contains six sensitivity specifications,True
9,Task M common-period performance remains profi...,True



FINAL AUG ASSESSMENT
1. The finalized AUG replication exhibits strong historical rolling OOS performance under the prescribed Channel WithDDControl framework.
2. Relative to the full-sample hindsight IS benchmark, OOS Sharpe, profit factor, win rate, expectancy, and payoff characteristics do not show a clear collapse in historical trade-level or risk-adjusted performance.
3. Task M shows that the strong historical OOS result is not unique to the original 4-year IS / 3-month OOS design within the tested T/tau range.
4. The evidence nevertheless contains material limitations: StpPct is persistently selected at the prescribed lower boundary, OOS profit is temporally concentrated, and a large share of net OOS P&L is attributed to session-gap bars.
5. The non-gap diagnostic remains historically profitable, but its materially weaker performance confirms that the magnitude of the headline AUG result is sensitive to session-gap attribution.
6. The relatively short AUG history and the limited 